# PatchTST 时序预测模型

本 notebook 展示 PatchTST 模型的架构、训练流程和评估指标。

**核心创新**：
- **Patching**：将时间序列切分为 patches（类似 ViT 处理图像），减少序列长度
- **Channel Independence**：每个变量独立建模，避免跨变量干扰
- **Transformer 编码器**：处理 patch embedding 序列，全局平均池化后预测

In [ ]:
import sys
sys.path.insert(0, '../../')

import torch
import numpy as np
import matplotlib.pyplot as plt

from models import PatchTSTModel, TimeSeriesDataset, Trainer
from models.trainer import resolve_device
from torch.utils.data import DataLoader, Subset
from pathlib import Path

device = resolve_device('auto')
print(f'Device: {device}')

DATA_DIR = Path('../../') / 'data' / 'processed'

## 1. 模型架构

```
输入 (batch, lookback, features)  -- features=7 for ETTh1
  ↓ permute → reshape: (batch×features, lookback, 1)  [Channel Independence]
  ↓ PatchEmbedding: Conv1d(1, d_model, kernel=patch_len, stride=stride)
  ↓   (lookback=96, patch_len=16, stride=8) → num_patches = (96-16)//8+1 = 11
  ↓ + PositionalEncoding (learnable)
  ↓ × N PatchTSTBlock
  │   ├─ MultiHeadAttention (d_model, n_heads)
  │   └─ FFN (d_model → d_ff → d_model)
  ↓ AdaptiveAvgPool1d(1) → Linear(d_model, horizon)
  ↓ reshape → permute
输出 (batch, horizon, features)
```

**关键设计**：
- 所有变量共享同一套权重（Channel Independence），相当于 batch 维度从 B 扩大到 B×C
- patch_len=16, stride=8 意味着相邻 patch 有 50% 重叠
- 使用 Conv1d 同时完成 patching 和 embedding

In [ ]:
ds = TimeSeriesDataset(DATA_DIR, 'ETTh1', 96, 'train')
print(f'数据集: ETTh1 h96')
print(f'  input_size: {ds.input_size}, target_idx: {ds.target_idx}')
print(f'  训练样本: {len(ds)}, X: {ds.X.shape}, Y: {ds.Y.shape}')

# 默认配置 (与正式实验一致)
model = PatchTSTModel(input_size=ds.input_size, d_model=64, n_heads=4,
                       n_layers=2, d_ff=128, patch_len=16, stride=8,
                       dropout=0.1, horizon=96)
params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'\n默认配置参数量: {params:,}')

# Patch 数量计算
num_patches = (96 - 16) // 8 + 1
print(f'patch_len=16, stride=8 → num_patches = (96-16)//8+1 = {num_patches}')

# 前向传播
x = torch.randn(2, 96, ds.input_size)
y = model(x)
print(f'前向传播: {x.shape} → {y.shape}')
print(f'  channel independence: batch {x.shape[0]} × {x.shape[2]} variables = {x.shape[0]*x.shape[2]} effective samples')

## 2. 快速训练与评估

In [ ]:
train_ds = TimeSeriesDataset(DATA_DIR, 'ETTh1', 96, 'train')
val_ds   = TimeSeriesDataset(DATA_DIR, 'ETTh1', 96, 'val')
test_ds  = TimeSeriesDataset(DATA_DIR, 'ETTh1', 96, 'test')

train_loader = DataLoader(Subset(train_ds, range(512)), batch_size=32, shuffle=True)
val_loader   = DataLoader(Subset(val_ds,   range(128)), batch_size=32)
test_loader  = DataLoader(test_ds, batch_size=32)

model = PatchTSTModel(input_size=ds.input_size, d_model=64, n_heads=4,
                       n_layers=2, d_ff=128, patch_len=16, stride=8,
                       dropout=0.1, horizon=96)
print(f'参数量: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

trainer = Trainer(model, device=device, lr=1e-3, weight_decay=1e-5, seed=42)
history = trainer.train(train_loader, val_loader, epochs=5, patience=10)
print(f'\n训练完成: {len(history["train_losses"])} epochs, best_val_loss={history["best_val_loss"]:.4f}')

## 3. 评估指标

In [ ]:
predictions, targets = trainer.predict(test_loader)
metrics = trainer.compute_metrics(predictions, targets, target_idx=ds.target_idx)

print('=== 全变量指标 ===')
print(f'  MSE:  {metrics["MSE"]:.4f}')
print(f'  MAE:  {metrics["MAE"]:.4f}')
print(f'  R²:   {metrics["R2"]:.4f}')
print(f'  MAPE: {metrics["MAPE"]:.2f}%')
print('\n=== 目标列指标 (OT) ===')
print(f'  MSE_target:  {metrics["MSE_target"]:.4f}')
print(f'  MAE_target:  {metrics["MAE_target"]:.4f}')
print(f'  R²_target:   {metrics["R2_target"]:.4f}')
print(f'  MAPE_target: {metrics["MAPE_target"]:.2f}%')

## 4. 可视化

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

# 损失曲线
ax = axes[0]
ax.plot(history['train_losses'], label='Train Loss')
ax.plot(history['val_losses'], label='Val Loss')
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE Loss')
ax.set_title('PatchTST Training')
ax.legend()
ax.grid(True, alpha=0.3)

# 预测 vs 真实
ax = axes[1]
pred_target = predictions[:64, :, ds.target_idx]
true_target = targets[:64, :, ds.target_idx]
ax.plot(true_target.flatten(), label='True', alpha=0.7)
ax.plot(pred_target.flatten(), label='Pred', alpha=0.7)
ax.set_xlabel('Time Step')
ax.set_ylabel('Value')
ax.set_title('PatchTST Prediction vs True (OT)')
ax.legend()
ax.grid(True, alpha=0.3)

# Patch embedding 可视化
ax = axes[2]
from models.patchtst import PatchEmbedding
patch_emb = PatchEmbedding(patch_len=16, stride=8, d_model=64, dropout=0.0)
x_single = torch.randn(1, 96, 1)  # 单变量
patches = patch_emb(x_single)  # (1, num_patches, d_model)
ax.imshow(patches[0].detach().T.numpy(), aspect='auto', cmap='RdBu_r')
ax.set_xlabel('Patch Index')
ax.set_ylabel('d_model Dimension')
ax.set_title(f'Patch Embeddings ({patches.shape[1]} patches × {patches.shape[2]} dims)')

plt.tight_layout()
plt.show()